In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd
from tqdm import tqdm
import zipfile
import glob
import os

In [2]:
download_dir = "\\".join((os.getcwd().split("\\")[:-1])+["data"])
os.makedirs(download_dir, exist_ok=True)
ruta = f"{download_dir}/csv"
os.makedirs(ruta, exist_ok=True)

In [3]:

options = webdriver.ChromeOptions()
options.add_experimental_option("prefs", {
    "download.default_directory": download_dir,  # aquí se guardarán las descargas
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
})

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get("https://dados.gov.br/dados/conjuntos-dados/arboviroses-dengue")
time.sleep(5)  # espera a que cargue todo

# Encuentra todos los bloques de recursos
blocks = driver.find_elements(By.XPATH, "//div[contains(@class,'row flex mb-5')]")

boton = driver.find_elements(By.ID, "btnCollapse")
boton = boton[2]
boton.click()
time.sleep(5)

for block in blocks:
    try:
        title = block.find_element(By.TAG_NAME, "h4").text.strip()
        formato = block.find_element(By.TAG_NAME, "span").text.strip()
        
        # Filtrar por año y formato CSV
        if any(str(year) in title for year in range(2020, 2027)) and formato == "CSV":
            print(f"Descargando: {title} ({formato})")
            button = block.find_element(By.XPATH, ".//button[contains(text(),'Acessar o recurso')]")
            button.click()
            time.sleep(2)  
    except Exception as e:
        print("Error en bloque:", e)

while True:
    noReady = [f for f in os.listdir(download_dir) if f.endswith(".crdownload")]
    #print(noReady)
    if len(noReady)==0:break
    time.sleep(10)

driver.quit()



In [4]:
def unzip_all(input_folder="data", output_folder="data/csv"):
    # Crear carpeta de salida si no existe
    os.makedirs(output_folder, exist_ok=True)

    # Buscar todos los archivos .zip en la carpeta data/zip
    zips = [f for f in os.listdir(input_folder) if f.endswith(".zip")]

    print(f"Encontrados {len(zips)} archivos ZIP en {input_folder}")

    for z in tqdm(zips, desc="Descomprimiendo ZIPs", leave=False):
        zip_path = os.path.join(input_folder, z)
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(output_folder)
        except Exception as e:
            print(f"Error al descomprimir {z}: {e}")

    print(f"✅ Todos los ZIPs fueron extraídos en {output_folder}")



In [5]:
def csvMerge(input_folder="data/csv", output_folder="data", filename="dengue_brasil.csv"):
    # Crear carpeta de salida si no existe
    os.makedirs(output_folder, exist_ok=True)

    # Buscar todos los archivos CSV en la carpeta data/csv
    files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if "prepro" in f.lower() and f.endswith(".csv")]

    print(f"Encontrados {len(files)} archivos CSV en {input_folder}")

    val = True
    for f in tqdm(files, desc="Fusionando CSV", leave=False):
        try:
            df = pd.read_csv(f)
            if val:
                DF = df
                val = False
            else:
                DF = pd.concat([DF, df], ignore_index=True)
        except Exception as e:
            print(f"Error leyendo {f}: {e}")

    # Guardar el CSV final
    output_path = os.path.join(output_folder, filename)
    DF.to_csv(output_path, index=False)
    print(f"CSV fusionado guardado en {output_path}")


variable edad 

se suara la variable NU_IDAE_N si es prefijo 4 se sua la edad, si no, se usa 0

In [6]:
def transformar_edad(valor):
    try:
        valor_str = str(valor)
        if valor_str.startswith("4"):
            # Tomar los dos últimos dígitos
            return int(valor_str[-2:])
        else:
            return 0
    except:
        return 0

toma_muestra

In [7]:
def clasificar_muestra(valor):
    try:
        v = int(valor)
        if v in [1, 2, 3]:
            return 1
        elif v == 4:
            return 2
        else:
            return 0  # nan
    except:
        return None


hemorragicos

In [8]:
def marcar_obito(valor):
    # Si es NaN → 0, si no → 1
    return 0 if pd.isna(valor) else 1




### PIPE  LINE

In [9]:
unzip_all()

Encontrados 0 archivos ZIP en data


✅ Todos los ZIPs fueron extraídos en data/csv


In [ ]:
# Lista de variables que quieres conservar
variables_sinan = [
    "CS_SEXO",
    "NU_IDADE_N",
    "SG_UF",
    "ID_MN_RESI",
    "SG_UF_NOT",
    "ID_MUNICIP",
    "ID_UNIDADE",
    "DT_SIN_PRI",
    "DIABETES",
    "HIPERTENSA",
    "ACIDO_PEPT",
    "RENAL",
    "AUTO_IMUNE",
    "HEPATOPAT",
    "CS_GESTANT",
    "DT_OBITO",
    "RESUL_PCR_",
    "CLASSI_FIN",
    "ALRM_SANG"
]+["SOROTIPO"]

# Ruta de los CSV originales
#ruta = "data/csv"
archivos = glob.glob(os.path.join(ruta, "*.csv"))

for archivo in archivos:
    try:
        # Leer solo las columnas necesarias
        df = pd.read_csv(archivo, usecols=variables_sinan, low_memory=False)

        df = df.drop_duplicates()
        
        # Crear columnas derivadas
        df["TOMA_MUESTRA"] = df["RESUL_PCR_"].apply(clasificar_muestra)
        df["DEFUNCION"] = df["DT_OBITO"].apply(marcar_obito)
        df["EDAD"] = df["NU_IDADE_N"].apply(transformar_edad)

        #RESUL_PCR_
        df['SOROTIPO'] = df['SOROTIPO'].fillna(5)
        df["SOROTIPO"] = df["SOROTIPO"].replace({0.0:5.0})

        # Eliminar columnas originales
        df = df.drop(columns=["RESUL_PCR_", "DT_OBITO", "NU_IDADE_N"], errors="ignore")

        nombre_base = os.path.basename(archivo).replace(".csv", "_prepro.csv")
        salida = os.path.join(ruta, nombre_base)
        df.to_csv(salida, index=False)

        print(f"Procesado: {archivo} → {salida}")
    except ValueError as e:
        print(f"Error en {archivo}: {e}")



In [11]:
csvMerge(download_dir+"/csv",download_dir+"/csv", download_dir+"/csv/dengue_brasil.csv")

Encontrados 0 archivos CSV en c:\Users\hormi\Documents\repositorios\dengue-epidemiology-mx-br\data/csv


UnboundLocalError: cannot access local variable 'DF' where it is not associated with a value